# Week 90, Day 3: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w090d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
walks = [{"id": i, "km": 3.0 + i} for i in range(1, 8)]
page_one = walks[0:3]
del walks[1]                      # someone deletes walk 2 while we read
page_two = walks[3:6]
print([w["id"] for w in page_one], [w["id"] for w in page_two])

> **Key idea.** Describe request bodies as schemas and check them in one function that reports every problem with its location; rule out booleans where integers are expected. Paginate with opaque, validated cursors that encode the last item seen, not offsets. Return every error in one envelope (code, message, details), mapping exceptions to statuses in one table, with 500 for anything unexpected. Register routes with summaries and stories and generate the reference from the registry. Version paths (`/v1/`), and deprecate old versions with headers before removing them.

### Exercise: A small schema checker

Write `check(data, schema, where="body")`: a list of problems. A schema is a dict with `type` (`"object"`, `"string"`, `"integer"`, `"number"`, `"boolean"` or `"array"`); for objects, `properties` (name to schema), `required` (names) and `additional` (default `True`; if `False`, unknown fields are problems); for strings, `min_length` and `max_length`; for numbers and integers, `minimum` and `maximum`; for arrays, `items` (a schema) and `min_items`. Problems are `"LOCATION: MESSAGE"` with messages `"expected TYPE"`, `"is required"`, `"is not allowed"`, `"must be at least N characters"`, `"must be at most N characters"`, `"must be at least N"`, `"must be at most N"` and `"needs at least N items"`. Locations are the field names joined by dots, with `[i]` for array items, starting with `where` (`body.stops[2]`). Integers and numbers never accept booleans; numbers accept ints.

In [ ]:
def check(data, schema, where="body"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("WALK = {\"type\": \"object\", \"additional\": False, \"required\": [\"day\", \"km\", \"minutes\"],\n        \"properties\": {\"day\": {\"type\": \"string\", \"min_length\": 10, \"max_length\": 10},\n                       \"km\": {\"type\": \"number\", \"minimum\": 0.1, \"maximum\": 100},\n                       \"minutes\": {\"type\": \"integer\", \"minimum\": 1, \"maximum\": 1440},\n                       \"stops\": {\"type\": \"array\", \"min_items\": 2, \"items\": {\"type\": \"string\", \"min_length\": 1}}}}\n\ndef test_schema():\n    \"every problem, with its place\"\n    assert check({\"day\": \"2026-11-14\", \"km\": 6, \"minutes\": 99, \"stops\": [\"Mill\", \"Weir\"]}, WALK) == []\n    assert check({\"day\": \"14/11\", \"km\": True, \"colour\": \"red\", \"stops\": [\"Mill\", 3, \"\"]}, WALK) == [\n        \"body.day: must be at least 10 characters\", \"body.km: expected number\", \"body.minutes: is required\",\n        \"body.stops[1]: expected string\", \"body.stops[2]: must be at least 1 characters\", \"body.colour: is not allowed\"]\n    assert check({\"day\": \"2026-11-14\", \"km\": 0, \"minutes\": 2000, \"stops\": [\"Mill\"]}, WALK) == [\n        \"body.km: must be at least 0.1\", \"body.minutes: must be at most 1440\", \"body.stops: needs at least 2 items\"]\n    assert check([1], WALK) == [\"body: expected object\"] and check(3, {\"type\": \"integer\"}, \"n\") == []")

### Exercise: Cursor pagination

Write `encode_cursor(last_id)`: the URL-safe base64 (padding removed) of the JSON `{"after": last_id}`. `decode_cursor(text)`: the id back, or `ValueError("bad cursor")` for anything that isn't a valid cursor with a whole-number `after`. And `page(rows, cursor=None, limit=3)`: for rows sorted by `id`, the next `limit` rows with `id` greater than the cursor's (all from the start if `None`), as `{"items": [...], "next": CURSOR_OR_NONE}`, where `next` is set only if more rows follow.

In [ ]:
import base64
import json

def encode_cursor(last_id):
    ___

def decode_cursor(text):
    ___

def page(rows, cursor=None, limit=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cursors():\n    \"stable pages\"\n    rows = [{\"id\": i} for i in range(1, 8)]\n    first = page(rows)\n    assert [r[\"id\"] for r in first[\"items\"]] == [1, 2, 3] and decode_cursor(first[\"next\"]) == 3\n    assert \"=\" not in first[\"next\"] and first[\"next\"] == encode_cursor(3)\n    del rows[1]\n    second = page(rows, first[\"next\"])\n    assert [r[\"id\"] for r in second[\"items\"]] == [4, 5, 6], \"nothing skipped after a deletion\"\n    third = page(rows, second[\"next\"])\n    assert [r[\"id\"] for r in third[\"items\"]] == [7] and third[\"next\"] is None\n    for bad in (\"!!\", encode_cursor(\"x\").replace(\"x\", \"y\"), base64.urlsafe_b64encode(b'{\"after\": true}').decode(), \"e30\"):\n        try:\n            decode_cursor(bad)\n        except ValueError as e:\n            assert str(e) == \"bad cursor\"\n        else:\n            raise AssertionError(bad)")

### Exercise: One error shape

Write `api_error(status, code, message, details=None)`: `(status, {"error": {"code": code, "message": message}})`, with `"details"` added only when given. Then `from_exception(exc)` using the `ERRORS` table (given, from exception class to `(status, code)`): find the first entry whose class the exception is an instance of, and return `api_error(status, code, str(exc))`; for anything else, `api_error(500, "internal", "something went wrong")`, never revealing the exception's text.

In [ ]:
class UnknownPlace(LookupError):
    pass

class DuplicateName(Exception):
    pass

ERRORS = [(UnknownPlace, (404, "not_found")), (DuplicateName, (409, "conflict")), (ValueError, (400, "invalid_request"))]

def api_error(status, code, message, details=None):
    ___

def from_exception(exc):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_errors():\n    \"one envelope\"\n    assert api_error(400, \"invalid_request\", \"bad body\", [\"body.km: expected number\"]) == (\n        400, {\"error\": {\"code\": \"invalid_request\", \"message\": \"bad body\", \"details\": [\"body.km: expected number\"]}})\n    assert api_error(404, \"not_found\", \"no such walk\") == (404, {\"error\": {\"code\": \"not_found\", \"message\": \"no such walk\"}})\n    assert from_exception(UnknownPlace(\"Moon\")) == (404, {\"error\": {\"code\": \"not_found\", \"message\": \"Moon\"}})\n    assert from_exception(DuplicateName(\"Ridge walk\"))[0] == 409 and from_exception(ValueError(\"km must be positive\"))[1][\"error\"][\"code\"] == \"invalid_request\"\n    assert from_exception(KeyError(\"secret_column\")) == (500, {\"error\": {\"code\": \"internal\", \"message\": \"something went wrong\"}})")

### Exercise: Routes that document themselves

Write a class `Registry` with a decorator method `route(method, path, summary="", stories=())` that records the handler with its details (in registration order) and returns the handler unchanged. `docs()` returns a Markdown table: a header `| Method | Path | Summary | Stories |`, a separator `|---|---|---|---|`, and a row per route with stories joined by `, ` (or `-`). `problems()` lists `"no summary: METHOD PATH"` and `"no story: METHOD PATH"` for each route, in order.

In [ ]:
class Registry:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_registry():\n    \"the reference, from the code\"\n    api = Registry()\n\n    @api.route(\"GET\", \"/v1/walks\", \"List walks, newest last, a page at a time\", [\"S5\"])\n    def list_walks():\n        return \"walks\"\n\n    @api.route(\"POST\", \"/v1/walks\", \"Log a walk\", [\"S5\"])\n    def log_walk():\n        pass\n\n    @api.route(\"GET\", \"/v1/health\")\n    def health():\n        pass\n\n    assert list_walks() == \"walks\", \"the handler is returned unchanged\"\n    assert api.docs() == (\"| Method | Path | Summary | Stories |\\n|---|---|---|---|\\n\"\n                          \"| GET | /v1/walks | List walks, newest last, a page at a time | S5 |\\n\"\n                          \"| POST | /v1/walks | Log a walk | S5 |\\n| GET | /v1/health |  | - |\")\n    assert api.problems() == [\"no summary: GET /v1/health\", \"no story: GET /v1/health\"]")

### Exercise: Versions in the path

Write `split_version(path, current="v1", deprecated=("v0",))`: for a path starting `/vN/...`, return `(version, rest, headers)`, where `rest` is the path without the version (starting with `/`) and `headers` is `{}` for the current version or `{"Deprecation": "true", "Sunset": "2027-06-30"}` for a deprecated one. Raise `LookupError("unsupported version vN")` for any other version, and `LookupError("no version in path")` if the path doesn't start with a version.

In [ ]:
import re

def split_version(path, current="v1", deprecated=("v0",)):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_versions():\n    \"change without breaking\"\n    assert split_version(\"/v1/walks\") == (\"v1\", \"/walks\", {})\n    assert split_version(\"/v0/walks/3\") == (\"v0\", \"/walks/3\", {\"Deprecation\": \"true\", \"Sunset\": \"2027-06-30\"})\n    for path, message in ((\"/v2/walks\", \"unsupported version v2\"), (\"/walks\", \"no version in path\"), (\"/version/1\", \"no version in path\")):\n        try:\n            split_version(path)\n        except LookupError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(path)")

### Bug hunt: A walk that was never shown

Offset pagination skips walk 4 when walk 2 is deleted between pages: it prints `[1, 2, 3] [5, 6, 7]`. Page by the last id seen instead, so it prints `[1, 2, 3] [4, 5, 6]`.

In [ ]:
walks = [{"id": i} for i in range(1, 8)]
first = walks[0:3]
del walks[1]
second = walks[3:6]
print([w["id"] for w in first], [w["id"] for w in second])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"after the last id\"\n    expect_output(\"[1, 2, 3] [4, 5, 6]\")")

### Bug hunt: True minutes

`isinstance(True, int)` is true, so a body with `"minutes": true` passes as one minute: it prints `[]`. Rule booleans out, so it prints `['minutes: expected integer']`.

In [ ]:
def check_minutes(body):
    if not isinstance(body.get("minutes"), int):
        return ["minutes: expected integer"]
    return []

print(check_minutes({"minutes": True}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bool is a subclass of int\"\n    expect_output(\"['minutes: expected integer']\")")

### Puzzle: A handler with the shared pieces

Arrange a handler that checks the body against its schema, reports problems in the shared envelope, and maps domain exceptions in one place. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # isinstance(True, int)
q = ___   # the location of the third stop in the request body, as the checker writes it
r = ___   # "offset" or "cursor": the pagination that can skip items when the list changes
s = ___   # the status from_exception gives a KeyError
t = ___   # the header that tells clients a version will be removed, and when

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"body.stops[2]\" and r == \"offset\", \"p, q and r\"\n    assert s == 500 and t == \"Sunset\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the walks API, version 1

Write `WalksAPI()` using the given helpers. It keeps walks in a list with ids from 1. `handle(method, path, body=None, query=None)` returns `(status, data, headers)`. Paths go through `split_version` (a `LookupError` becomes `404 not_found` with its message). Routes: `GET /walks` with an optional `cursor` and `limit` (1 to 50, default 10) in `query` returns `200 {"items", "next"}` from `page`; `POST /walks` checks `body` against `WALK_SCHEMA` (problems give `400 invalid_request` with message `"The request body has problems."` and the details) and returns `201` with the stored walk including its `id`; `GET /walks/N` returns `200` and the walk or `404 not_found` (`"no such walk"`). A bad cursor or limit is `400 invalid_request` with its message (`"bad cursor"`, `"limit must be 1 to 50"`). Another method on a known path is `405 method_not_allowed` (`"method not allowed"`); an unknown path `404 not_found` (`"no such route"`). Deprecation headers from `split_version` are returned on every response for that version; otherwise headers are `{}`.

In [ ]:
import base64
import json
import re

def check(data, schema, where="body"):
    kind = schema["type"]
    ok = {"object": lambda v: isinstance(v, dict), "string": lambda v: isinstance(v, str),
          "integer": lambda v: isinstance(v, int) and not isinstance(v, bool),
          "number": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)}[kind]
    if not ok(data):
        return [f"{where}: expected {kind}"]
    problems = []
    if kind == "object":
        for name, sub in schema.get("properties", {}).items():
            if name in data:
                problems += check(data[name], sub, f"{where}.{name}")
            elif name in schema.get("required", []):
                problems.append(f"{where}.{name}: is required")
        if schema.get("additional", True) is False:
            problems += [f"{where}.{n}: is not allowed" for n in data if n not in schema.get("properties", {})]
    elif kind in ("integer", "number"):
        if "minimum" in schema and data < schema["minimum"]:
            problems.append(f"{where}: must be at least {schema['minimum']}")
        if "maximum" in schema and data > schema["maximum"]:
            problems.append(f"{where}: must be at most {schema['maximum']}")
    return problems

WALK_SCHEMA = {"type": "object", "additional": False, "required": ["day", "km", "minutes"],
               "properties": {"day": {"type": "string"}, "km": {"type": "number", "minimum": 0.1},
                              "minutes": {"type": "integer", "minimum": 1, "maximum": 1440}}}

def encode_cursor(last_id):
    return base64.urlsafe_b64encode(json.dumps({"after": last_id}).encode()).decode().rstrip("=")

def decode_cursor(text):
    try:
        data = json.loads(base64.urlsafe_b64decode(text + "=" * (-len(text) % 4)))
    except (ValueError, TypeError):
        raise ValueError("bad cursor")
    after = data.get("after") if isinstance(data, dict) else None
    if not isinstance(after, int) or isinstance(after, bool):
        raise ValueError("bad cursor")
    return after

def page(rows, cursor=None, limit=3):
    after = decode_cursor(cursor) if cursor else None
    remaining = [r for r in rows if after is None or r["id"] > after]
    items = remaining[:limit]
    return {"items": items, "next": encode_cursor(items[-1]["id"]) if len(remaining) > limit else None}

def api_error(status, code, message, details=None):
    body = {"code": code, "message": message}
    if details is not None:
        body["details"] = details
    return status, {"error": body}

def split_version(path, current="v1", deprecated=("v0",)):
    m = re.match(r"/(v\d+)(/.*)?$", path)
    if not m:
        raise LookupError("no version in path")
    version, rest = m.group(1), m.group(2) or "/"
    if version == current:
        return version, rest, {}
    if version in deprecated:
        return version, rest, {"Deprecation": "true", "Sunset": "2027-06-30"}
    raise LookupError(f"unsupported version {version}")

class WalksAPI:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _walk(i):\n    return {\"day\": f\"2026-01-{i:02d}\", \"km\": 3.0 + i, \"minutes\": 40 + i}\n\ndef test_walks():\n    \"create, page, fetch\"\n    api = WalksAPI()\n    for i in range(1, 6):\n        status, walk, headers = api.handle(\"POST\", \"/v1/walks\", _walk(i))\n        assert status == 201 and walk[\"id\"] == i and headers == {}\n    status, first, _ = api.handle(\"GET\", \"/v1/walks\", query={\"limit\": \"2\"})\n    assert [w[\"id\"] for w in first[\"items\"]] == [1, 2] and first[\"next\"]\n    status, second, _ = api.handle(\"GET\", \"/v1/walks\", query={\"limit\": \"2\", \"cursor\": first[\"next\"]})\n    assert [w[\"id\"] for w in second[\"items\"]] == [3, 4]\n    assert api.handle(\"GET\", \"/v1/walks/5\")[:2] == (200, {\"id\": 5, **_walk(5)})\n    status, data, headers = api.handle(\"GET\", \"/v0/walks/5\")\n    assert status == 200 and headers == {\"Deprecation\": \"true\", \"Sunset\": \"2027-06-30\"}\n\ndef test_errors():\n    \"one envelope everywhere\"\n    api = WalksAPI()\n    status, data, _ = api.handle(\"POST\", \"/v1/walks\", {\"day\": \"2026-01-01\", \"km\": True, \"extra\": 1})\n    assert status == 400 and data == {\"error\": {\"code\": \"invalid_request\", \"message\": \"The request body has problems.\",\n                                                \"details\": [\"body.km: expected number\", \"body.minutes: is required\", \"body.extra: is not allowed\"]}}\n    assert api.handle(\"GET\", \"/v1/walks\", query={\"cursor\": \"!!\"})[:2] == (400, {\"error\": {\"code\": \"invalid_request\", \"message\": \"bad cursor\"}})\n    assert api.handle(\"GET\", \"/v1/walks\", query={\"limit\": \"500\"})[1][\"error\"][\"message\"] == \"limit must be 1 to 50\"\n    assert api.handle(\"GET\", \"/v1/walks/9\")[:2] == (404, {\"error\": {\"code\": \"not_found\", \"message\": \"no such walk\"}})\n    assert api.handle(\"DELETE\", \"/v1/walks\")[:2] == (405, {\"error\": {\"code\": \"method_not_allowed\", \"message\": \"method not allowed\"}})\n    assert api.handle(\"GET\", \"/v2/walks\")[:2] == (404, {\"error\": {\"code\": \"not_found\", \"message\": \"unsupported version v2\"}})\n    assert api.handle(\"GET\", \"/v1/routes\")[1][\"error\"][\"message\"] == \"no such route\"")

## Warm-up

Walk 4 was never shown: deleting an item on page one shifted everything after it, and "items 3 to 5" now means different items. Pages counted by **position** slip whenever the list changes underneath them. Pages that remember the **last item seen** don't.

## Checking requests in one place

Every handler that accepts JSON needs the same checks: required fields present, of the right types, within ranges, and nothing unexpected. Written by hand in each handler, they drift apart. Instead, describe each request body as a **schema** (data, not code) and check it with one function that returns **every** problem with its **location** (`stops[2]: expected string`), so an API client can fix them all at once. Real projects use JSON Schema (a standard) or libraries such as Pydantic; today's small version shows what they do. One Python trap: `bool` is a subclass of `int`, so `isinstance(True, int)` is true, and a schema checker must rule booleans out explicitly.

## Pages that don't slip

`GET /walks` can't return a walker's whole history in one response. **Offset** pagination (`?offset=20&limit=10`) is simple and slips, as the warm-up showed. **Cursor** pagination returns, with each page, a `next` token encoding "after walk 23"; the next request asks for walks with `id > 23`, which is stable however the list changes. Make the cursor **opaque** (base64 of a little JSON) so clients don't build them by hand, and **validate** it on the way in: a cursor is user input too.

## One error shape

Every error, from every route, in one envelope:

```json
{"error": {"code": "invalid_request", "message": "The request body has problems.", "details": ["days: must be at least 1"]}}
```

with a machine-readable `code` (clients switch on it), a human `message`, and optional `details`. Map domain exceptions to statuses and codes in **one table** (`UnknownPlace` to 404 `not_found`, `DuplicateName` to 409 `conflict`, `ValueError` to 400 `invalid_request`) consulted by one function, so a new exception is one line, and nothing unexpected leaks a traceback (500 `internal`).

## Documentation and versions

Register each route with its method, path, a one-line summary and the stories it serves, using a decorator (Week 60), and generate the **reference documentation** from the registry: a table that can't go out of date, because it's made from the code. A test can then insist that every route has a summary and a story.

Put a **version** in the path (`/v1/walks`). When a change would break clients (renaming a field, changing a type), add `/v2/` and keep `/v1/` working for a while, marking its responses with a `Deprecation` header and a date in `Sunset`, so clients have time to move.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **One schema per request body**, checked in one function, with tests for each kind of problem.
2. **Cursors** for every list that can grow.
3. **One error envelope** and one exception table; make sure a 500 never shows details.
4. **A registry** that writes your API (or command) reference, and a test that every entry has a summary and a story.
5. **Merge** and update the burndown.

## Before you go

The build days of the last sprint are over. Tomorrow is the tests day, focused on the API as a whole: contracts, coverage of every route and status, and end-to-end scenarios.